# [실습 3] Attention Mask, KV Cache, generate() 최적화 이해


## 실습 개요

> "작은 LLM에서도 attention mask와 KV cache 구조를 직접 확인할 수 있을까?"

이번 실습에서는 `HuggingFaceTB/SmolLM2-135M-Instruct`을 사용해 decoder-only 생성 모델의 입력 구조를 확인합니다.  
`attention_mask`, `past_key_values`, `generate()` 파라미터를 직접 다루며 강의에서 다룬 Attention 및 추론 최적화 개념을 실습합니다.


## 실습 목표

1. Padding mask와 causal language model 입력 구조를 이해한다.
2. `past_key_values`가 KV cache 역할을 한다는 점을 확인한다.
3. `generate()`의 greedy/sampling 설정 차이를 비교한다.
4. `torch_dtype`, `device_map`, `low_cpu_mem_usage` 같은 로딩 옵션의 의미를 정리한다.
5. [TODO] 서로 다른 생성 전략을 함수로 비교한다.


## 데이터 출처

- 데이터셋: 실습용 프롬프트 직접 구성
- 모델: `HuggingFaceTB/SmolLM2-135M-Instruct` (Hugging Face Hub)


## 목차

1. 라이브러리 임포트 및 모델 로드
2. attention_mask와 padding 확인
3. Causal LM 출력과 logits 확인
4. KV cache 구조 확인
5. generate() 전략 비교
6. 로딩 최적화 옵션 정리
7. [TODO] 생성 전략 비교 함수 만들기


---
## 1. 라이브러리 임포트 및 모델 로드


In [9]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

checkpoint = 'HuggingFaceTB/SmolLM2-135M-Instruct'
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForCausalLM.from_pretrained(checkpoint)
model.eval()

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = tokenizer.eos_token_id

print(f'모델 클래스: {type(model).__name__}')
print(f'pad_token: {tokenizer.pad_token!r}')

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5092.11it/s]


모델 클래스: LlamaForCausalLM
pad_token: '<|im_end|>'


---
## 2. attention_mask와 padding 확인

배치 입력에서는 문장 길이가 다를 수 있으므로 padding이 필요합니다.  
`attention_mask`는 실제 토큰 위치를 1, padding 위치를 0으로 표시합니다.


In [10]:
prompts = [
    'Transformers are useful because',
    'I love my',
]

batch = tokenizer(prompts, padding=True, return_tensors='pt')

print('input_ids shape:', tuple(batch['input_ids'].shape))
print('attention_mask shape:', tuple(batch['attention_mask'].shape))
print('\nattention_mask:')
print(batch['attention_mask'])

for i, ids in enumerate(batch['input_ids']):
    print(f'[{i}]', tokenizer.decode(ids))

input_ids shape: (2, 5)
attention_mask shape: (2, 5)

attention_mask:
tensor([[1, 1, 1, 1, 1],
        [1, 1, 1, 0, 0]])
[0] Transformers are useful because
[1] I love my<|im_end|><|im_end|>


---
## 3. Causal LM 출력과 logits 확인

Causal LM은 각 위치에서 다음 토큰 분포를 예측합니다.  
출력 `logits`의 shape은 `(batch, seq_len, vocab_size)`입니다.


In [11]:
with torch.no_grad():
    outputs = model(**batch)

logits = outputs.logits
print('logits shape:', tuple(logits.shape))

last_token_logits = logits[:, -1, :]
next_token_ids = torch.argmax(last_token_logits, dim=-1)

for prompt, token_id in zip(prompts, next_token_ids):
    print(f'입력: {prompt!r}')
    print(f'greedy next token: {tokenizer.decode([token_id.item()])!r}')

logits shape: (2, 5, 49152)
입력: 'Transformers are useful because'
greedy next token: ' they'
입력: 'I love my'
greedy next token: '.'


---
## 4. KV cache 구조 확인

`use_cache=True`로 forward를 실행하면 `past_key_values`가 반환됩니다.  
이는 이전 토큰들의 Key/Value를 저장한 캐시이며, 다음 토큰 생성 시 중복 계산을 줄이는 데 사용됩니다.


In [15]:
single = tokenizer('Attention cache helps generation', return_tensors='pt')

with torch.no_grad():
    cached_outputs = model(**single, use_cache=True)

past = cached_outputs.past_key_values
print('레이어 수:', len(past))

past_layers = list(past)
first_layer_key, first_layer_value, _ = past_layers[0]

print('첫 레이어 key shape:', tuple(first_layer_key.shape))
print('첫 레이어 value shape:', tuple(first_layer_value.shape))

레이어 수: 30
첫 레이어 key shape: (1, 3, 4, 64)
첫 레이어 value shape: (1, 3, 4, 64)


---
## 5. generate() 전략 비교

`generate()`는 greedy decoding, sampling, 길이 제한 등 다양한 옵션을 제공합니다.  
같은 프롬프트라도 `do_sample`, `temperature`, `top_k`, `max_new_tokens` 조합에 따라 결과가 달라질 수 있습니다.


In [16]:
prompt = 'The main benefit of KV cache is'
inputs = tokenizer(prompt, return_tensors='pt')

with torch.no_grad():
    greedy_ids = model.generate(**inputs, max_new_tokens=20, do_sample=False)
    sampled_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=True,
        temperature=0.8,
        top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )

print('[Greedy]')
print(tokenizer.decode(greedy_ids[0], skip_special_tokens=True))
print('\n[Sampling]')
print(tokenizer.decode(sampled_ids[0], skip_special_tokens=True))

[Greedy]
The main benefit of KV cache is that it allows for faster access to data, especially when the cache is full. This is particularly useful

[Sampling]
The main benefit of KV cache is that it can be placed in any cache location, allowing for dynamic layout and layout reconfiguration. This


---
## 6. 로딩 최적화 옵션 정리

대형 모델을 로드할 때는 다음 옵션을 함께 고려합니다.

| 옵션 | 의미 |
|------|------|
| `torch_dtype=torch.float16` 또는 `torch.bfloat16` | GPU 메모리 사용량 절감 및 속도 개선 |
| `device_map="auto"` | 여러 GPU/CPU에 레이어 자동 배치 |
| `low_cpu_mem_usage=True` | 로딩 중 CPU 메모리 피크 감소 |
| `load_in_4bit`, `load_in_8bit` | bitsandbytes 기반 양자화 로딩 |

SmolLM2-135M-Instruct는 작아서 필수는 아니지만, 실제 대형 LLM을 다룰 때 중요한 옵션입니다.


In [17]:
# 대형 모델 로딩 시 사용하는 패턴 예시입니다. 이 셀은 실행하지 않아도 됩니다.
example_loading_kwargs = {
    'torch_dtype': 'torch.float16 또는 torch.bfloat16',
    'device_map': 'auto',
    'low_cpu_mem_usage': True,
}

example_loading_kwargs

{'torch_dtype': 'torch.float16 또는 torch.bfloat16',
 'device_map': 'auto',
 'low_cpu_mem_usage': True}

---
### [TODO] 생성 전략 비교 함수 만들기

프롬프트와 생성 옵션을 받아 `model.generate()`로 텍스트를 생성하고, 생성된 토큰 ID를 문자열로 디코딩합니다.

`model.generate(**model_inputs, ...)`에 바로 넘길 수 있도록 토크나이저 결과를 PyTorch 텐서로 만들어야 합니다. `tokenizer()` 호출의 `return_tensors` 값을 `'pt'`로 지정하세요. 생성 옵션과 디코딩 코드는 그대로 사용합니다.


In [18]:
def generate_text(prompt, do_sample=False, temperature=1.0, top_k=50, max_new_tokens=20):
    model_inputs = tokenizer(
        prompt,
        return_tensors='pt',  # TODO: PyTorch 텐서로 반환되도록 수정
    )

    with torch.no_grad():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_k=top_k,
            pad_token_id=tokenizer.eos_token_id,
        )

    decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    return decoded


In [19]:
todo_prompt = 'Flash Attention improves'

greedy_text = generate_text(todo_prompt, do_sample=False, max_new_tokens=15)
sampling_text = generate_text(todo_prompt, do_sample=True, temperature=0.9, top_k=20, max_new_tokens=15)

print('[Greedy]')
print(greedy_text)
print('\n[Sampling]')
print(sampling_text)
print('\n문자열 타입 확인:', isinstance(greedy_text, str), isinstance(sampling_text, str))

[Greedy]
Flash Attention improves the overall flow and clarity of the text, making it easier for readers to

[Sampling]
Flash Attention improves the overall lighting of the setting, emphasizing the contrast between the lush foliage and

문자열 타입 확인: True True
